In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../../.env")
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
ml = Path("../../data/processed/ml")

# Attributs constants par commune + stats de prix (LOG = ln, comme np.log1p)
ref = pd.read_sql("""
    SELECT code_commune_geo,
           MAX(nom_commune_geo)             AS nom,
           MAX(`Code departement`)          AS code_departement,
           AVG(codeRegion)                  AS codeRegion,
           AVG(population_geo)              AS population_geo,
           AVG(longitude)                  AS longitude,
           AVG(latitude)                   AS latitude,
           AVG(revenu_median)              AS revenu_median,
           COUNT(*)                         AS n_ventes,
           AVG(LOG(1 + `Valeur fonciere`))  AS mean_log_prix
    FROM transactions
    GROUP BY code_commune_geo
""", con=engine)

print("Communes :", ref.shape)
ref.head()


Communes : (29805, 10)


,code_commune_geo,nom,code_departement,codeRegion,population_geo,longitude,latitude,revenu_median,n_ventes,mean_log_prix
0,01426,Val-Revermont,01,84.0,2488.0,5.3537,46.2816,21034.8,206,12.156683
1,01065,Buellas,01,84.0,1899.0,5.1468,46.2116,23229.0,129,12.539232
2,01254,Montagnat,01,84.0,2186.0,5.2776,46.1720,24781.3,121,12.655421
3,01344,Saint-Denis-lès-Bourg,01,84.0,6160.0,5.1825,46.2097,23351.5,425,12.342523
4,01301,Polliat,01,84.0,2741.0,5.1355,46.2504,22589.5,263,12.366956


In [2]:
# Moyenne globale du log-prix (pour le lissage)
global_log = pd.read_sql(
    "SELECT AVG(LOG(1 + `Valeur fonciere`)) AS g FROM transactions", engine
)["g"][0]

m = 20  # même lissage qu'à l'entraînement
ref["prix_median_commune"] = (
    ref["n_ventes"] * ref["mean_log_prix"] + m * global_log
) / (ref["n_ventes"] + m)

ref.to_parquet(ml / "communes_ref.parquet")
print("Table communes sauvegardée ✅")


Table communes sauvegardée ✅


In [3]:
# Médiane du prix par département (2 colonnes seulement -> léger)
dd = pd.read_sql("SELECT `Code departement` AS d, `Valeur fonciere` AS v FROM transactions", engine)
prix_dept = dd.groupby("d")["v"].median()

# Médiane du terrain des maisons (pour l'imputation)
tt = pd.read_sql("SELECT `Surface terrain` AS t FROM transactions WHERE `Type local`='Maison'", engine)
median_terrain = float(tt["t"].median())

joblib.dump(prix_dept,      ml / "prix_dept.pkl")
joblib.dump(median_terrain, ml / "median_terrain.pkl")
joblib.dump(float(global_log), ml / "global_log.pkl")   # valeur de repli
print("Artefacts sauvegardés ✅ | median_terrain =", median_terrain)


Artefacts sauvegardés ✅ | median_terrain = 501.0


Ce que ça fait :

communes_ref.parquet : pour chaque commune, ses attributs géo + son niveau de prix lissé → le dashboard remplira ces features automatiquement dès que l'utilisateur choisit une commune.
prix_dept.pkl / median_terrain.pkl : les tables apprises, indispensables pour reproduire le feature engineering à l'identique.
💡 Petite honnêteté méthodo : ici on calcule les encodages sur toutes les données (pas juste le train). Pour un dashboard de démo c'est parfaitement acceptable (ce sont des tables de référence, pas une évaluation). Le modèle, lui, reste celui entraîné proprement.